# H100 네 장에서 Gemma 4와 DFlash로 답변 생성하기

먼저 Gemma 4 31B IT가 혼자 답변을 쓰는 시간을 잰다. 이어서 **DFlash**라는 작은 초안 모델이 다음 토큰 여러 개를 제안하게 한다. 최종 토큰은 두 경우 모두 Gemma가 결정한다. 같은 질문을 보내 **한 사람이 기다리는 시간**과 **서버 전체가 1초 동안 생성한 토큰 수**를 각각 비교한다.

| 발표에서 본 내용 | 여기서 할 일 |
| --- | --- |
| 4.2 가중치와 모델 설정 | 로컬 모델 디렉터리를 열어 가중치의 형식과 초안의 블록 크기를 확인한다. |
| 4.3 추론 엔진 | H100 네 장으로 서버 하나를 띄우고 한국어 답변을 받는다. |
| 4.5 부하 테스트 | 한 번에 하나씩 보낸 경우와 최대 네 개씩 보낸 경우를 따로 비교한다. |

[4장 발표 자료](chapter-04.html) · [기존 H100 실습](h100.ipynb) · [기존 B300 실습](b300.ipynb)

## 1. GPU와 모델 준비

**실행 환경**은 H100 네 장이 연결된 Linux 서버 한 대다. 같은 머신에서 노트북과 서버 터미널을 연다. 이 노트북의 명령과 응답 필드는 **vLLM 0.34.0** 기준이다. 호환되는 NVIDIA 드라이버를 갖춘 Python 3.12 환경에 vLLM, JupyterLab, pandas가 필요하다. 새 환경을 준비할 때는 터미널에서 다음 명령을 쓸 수 있다.

```bash
uv venv --python 3.12
source .venv/bin/activate
uv pip install vllm==0.34.0 jupyterlab pandas --torch-backend=auto
vllm download-kernels
jupyter lab
```

노트북은 이 환경의 Python 커널로 연다. **2절과 5절에서 멈추고**, 셀이 출력한 명령을 별도 터미널에서 실행한다. 나머지 셀은 위에서 아래로 실행한다.

**모델 경로**는 아래 두 줄을 수정한다. `TARGET`은 Google Gemma 4 31B IT의 가중치, 토크나이저, 대화 템플릿이 있는 폴더다. `DRAFT`는 **이 대상 모델용** DFlash 체크포인트 폴더다. 두 폴더 모두 Hugging Face 형식의 로컬 파일을 사용한다. 예시 이름은 Z-Lab 배포본 기준이다. 셀은 경로에 있는 설정을 읽는다.

[vLLM GPU 설치](https://docs.vllm.ai/en/v0.34.0/getting_started/installation/gpu/) · [Gemma 모델 카드](https://huggingface.co/google/gemma-4-31B-it) · [DFlash 모델 카드](https://huggingface.co/z-lab/gemma-4-31B-it-DFlash)

In [ ]:
import json
import shlex
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

import pandas as pd
import requests
from prometheus_client.parser import text_string_to_metric_families

TARGET = Path("/models/gemma-4-31B-it")
DRAFT = Path("/models/gemma-4-31B-it-DFlash")

print("vLLM", version("vllm"))
print(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"],
    text=True,
))
target_config = json.loads((TARGET / "config.json").read_text())
draft_config = json.loads((DRAFT / "config.json").read_text())
print("대상 구조:", target_config["architectures"])
print("초안 구조:", draft_config["architectures"], "/ 블록 크기:", draft_config["block_size"])

## 2. Gemma 단독 서버 실행

**서버**는 요청을 받을 때마다 답을 돌려주는 프로그램이다. **추론 엔진**인 vLLM은 서버 안에서 모델을 실행하고 어떤 요청을 함께 계산할지 정한다. **TP 4**는 네 GPU가 모델 한 개의 행렬 계산을 나눈다는 뜻이다. **BF16**은 가중치 한 값을 2바이트에 담는 수 형식이다.

아래 셀은 실행할 명령을 **출력**한다. 명령 전체를 같은 머신의 터미널에 붙여 넣는다. 서버 로그에 `Application startup complete`가 보이면 다음 셀로 이동한다. 터미널은 열어 둔다.

두 구성 모두 문맥 길이 4,096토큰, 텍스트 입력, 같은 대상 어텐션 백엔드를 쓴다. *어텐션*은 답을 만들 때 앞의 어느 토큰을 많이 참고할지 계산하는 연산이다. *백엔드*는 이 계산을 수행하는 구현이다. 반복 질문의 입력 계산을 건너뛰는 **접두사 캐싱**은 끈다. 요청 안에서 이전 토큰의 키와 값(KV)을 재사용하는 캐시는 계속 쓴다.

In [ ]:
base_url = "http://127.0.0.1:8000"
server = [
    sys.executable, "-m", "vllm.entrypoints.cli.main", "serve", str(TARGET),
    "--host", "127.0.0.1", "--tensor-parallel-size", "4",
    "--dtype", "bfloat16", "--max-model-len", "4096",
    "--language-model-only", "--disable-prefix-caching",
    "--attention-backend", "TRITON_ATTN", "--enable-per-request-metrics",
]
print(shlex.join(server))

### 질문 하나 보내기

`messages`에 사용자 질문을 넣는다. `temperature=0`이면 각 단계에서 가장 점수가 높은 토큰을 선택한다. `max_tokens=128`은 답변 길이의 상한이다. 답이 일찍 끝나면 실제 토큰 수는 더 작다. 아래 질문의 정답은 7개다. 문장이 한국어인지, 개수가 맞는지 먼저 읽어 본다.

In [ ]:
def ask(question):
    start = perf_counter()
    response = requests.post(f"{base_url}/v1/chat/completions", json={
        "model": str(TARGET),
        "messages": [{"role": "user", "content": question}],
        "temperature": 0,
        "max_tokens": 128,
        "chat_template_kwargs": {"enable_thinking": False},
    })
    elapsed_ms = (perf_counter() - start) * 1000
    response.raise_for_status()
    body = response.json()
    choice = body["choices"][0]
    metrics = body["metrics"]
    return {
        "답변": choice["message"]["content"],
        "종료": choice["finish_reason"],
        "입력 토큰": body["usage"]["prompt_tokens"],
        "출력 토큰": body["usage"]["completion_tokens"],
        "TTFT ms": metrics["time_to_first_token_ms"],
        "TPOT ms": metrics["mean_itl_ms"],
        "큐 대기 ms": metrics["queue_time_ms"],
        "응답 ms": elapsed_ms,
    }

check_question = "사과가 12개 있다. 5개를 먹었다. 남은 개수를 한국어 한 문장으로 답해 줘."
baseline_answer = ask(check_question)
print(baseline_answer["답변"])
print("입력 토큰:", baseline_answer["입력 토큰"], "/ 출력 토큰:", baseline_answer["출력 토큰"])

## 3. 지표와 질문 준비

| 지표 | 어디서 어디까지 재나? |
| --- | --- |
| 큐 대기 (ms) | 엔진의 대기열에 들어간 때부터 스케줄러가 실행을 시작한 때까지 |
| 서버 TTFT (ms) | 스케줄 시작부터 첫 출력 토큰이 생성될 때까지. 기존 Qwen 노트의 클라이언트 TTFT는 요청을 전송할 때부터 잰다. |
| TPOT (ms/토큰) | 첫 출력 토큰부터 마지막 토큰까지의 시간 ÷ (출력 토큰 수 − 1). 한 토큰으로 끝나면 값이 없다. |
| 응답 시간 (ms) | 노트북이 요청을 보내기 직전부터 전체 응답을 받을 때까지 |
| 출력 처리량 (토큰/초) | 모든 요청이 생성한 토큰 수의 합 ÷ 첫 요청을 보내기 직전부터 모든 답을 받을 때까지의 경과 시간 |

**부하 테스트**에서는 여러 질문을 실제로 보낸다. 동시 요청 수가 1이면 앞 답변을 받은 후 다음 질문을 보낸다. 4이면 최대 네 요청이 겹친다. 하나가 끝나면 다음 질문을 보낸다.

한국어 질문 8개에 설명, 요약, 계산, 코드 작성을 섞었다. 각 조건에서 먼저 동시 요청 수만큼 준비 요청을 보낸다(**워밍업**). 이후 같은 여덟 질문을 같은 순서로 세 차례 보내 24건을 측정한다. 작은 스터디용 부하이므로 느린 요청 한두 개도 상위 지연값을 바꿀 수 있다. [vLLM 요청별 측정값](https://docs.vllm.ai/en/v0.34.0/features/per_request_metrics/)

In [ ]:
questions = [
    "모델이 첫 토큰을 만드는 프리필 과정을 두 문장으로 설명해 줘.",
    "입력 128토큰과 출력 32토큰을 합치면 몇 토큰인지 계산 과정을 써 줘.",
    "다음 문장을 한 문장으로 요약해 줘. 서버가 요청을 받았다. 요청이 대기열에 들어갔다. GPU가 차례로 처리했다.",
    "파이썬으로 리스트의 양수만 더하는 함수를 작성하고 예시를 하나 들어 줘.",
    "질문 두 개가 동시에 도착했을 때 연속 배칭이 어떻게 동작하는지 예를 들어 줘.",
    "정수 1부터 10까지 더한 값을 구하고 풀이를 두 문장으로 써 줘.",
    "클라이언트가 측정한 응답 시간에 포함되는 일을 순서대로 세 가지 적어 줘.",
    "출력 토큰 240개를 3초에 생성한 서버의 처리량을 단위와 함께 설명해 줘.",
]

def measure(concurrency, counters=None):
    with ThreadPoolExecutor(max_workers=concurrency) as pool:
        list(pool.map(ask, questions[:concurrency]))
        if counters:
            before = counters()
        start = perf_counter()
        answers = list(pool.map(ask, questions * 3))
        seconds = perf_counter() - start

    rows = pd.DataFrame(answers)
    summary = {
        "동시 요청": concurrency,
        "요청 수": len(rows),
        "입력 토큰 합": rows["입력 토큰"].sum(),
        "출력 토큰 합": rows["출력 토큰"].sum(),
        "출력 토큰/초": rows["출력 토큰"].sum() / seconds,
        "큐 대기 p50 ms": rows["큐 대기 ms"].median(),
        "TTFT p50 ms": rows["TTFT ms"].median(),
        "TPOT p50 ms": rows["TPOT ms"].median(),
        "응답 p95 ms": rows["응답 ms"].quantile(0.95),
    }
    if counters:
        after = counters()
        proposed = after["제안"] - before["제안"]
        accepted = after["채택"] - before["채택"]
        summary["제안 토큰"] = proposed
        summary["채택 토큰"] = accepted
        summary["채택률 %"] = 100 * accepted / proposed if proposed else float("nan")
    return summary

## 4. Gemma 단독 기준값 측정

**기준값**은 설정을 바꾸기 전에 얻은 값이다. 먼저 요청 수가 각 행에서 24인지 읽는다. 이어서 동시 요청 1과 4에서 처리량과 응답 시간이 어떻게 달라지는지 본다. **p50**은 요청을 빠른 순서로 놓았을 때 가운데에 해당하는 값이다. **p95**는 약 95%의 요청이 그 시간 이내에 끝나는 경계다.

In [ ]:
baseline = pd.DataFrame([measure(1), measure(4)]).set_index("동시 요청")
baseline.round(2)

## 5. DFlash를 연결하고 같은 질문 보내기

기준 서버를 실행한 **터미널에서 Ctrl+C**를 누른다. 서버가 종료되어 셸 입력이 다시 가능해지면 아래 셀이 출력한 새 명령을 같은 터미널에서 실행한다. **노트북 커널은 계속 열어 둔다.** 4절의 측정값이 커널 메모리에 있다.

DFlash는 Gemma가 이미 계산한 중간값을 보고 여러 위치의 초안을 한꺼번에 제안한다. Gemma가 앞에서부터 검증한다. 예를 들어 초안 A·B·C에서 앞의 A·B만 맞으면, A·B를 채택하고 세 번째 위치는 Gemma의 토큰으로 채운다.

아래 예의 블록에는 이미 정해진 토큰 한 자리가 포함된다. Z-Lab 31B용의 블록 크기는 16이므로 후보는 최대 **15개**다. 같은 대상용 Red Hat 배포본은 블록 크기 8에 후보 7개다. 이 두 체크포인트의 설정에서 후보 수를 읽어 적용한다. 대상 모델의 어텐션 계산은 그대로 두고, 초안 모델의 계산에는 FlashAttention을 지정한다. [Z-Lab 모델 카드](https://huggingface.co/z-lab/gemma-4-31B-it-DFlash) · [Red Hat 모델 카드](https://huggingface.co/RedHatAI/gemma-4-31B-it-speculator.dflash)

In [ ]:
speculation = {
    "method": "dflash",
    "model": str(DRAFT),
    "num_speculative_tokens": draft_config["block_size"] - 1,
    "attention_backend": "FLASH_ATTN",
}
print(shlex.join(server + ["--speculative-config", json.dumps(speculation)]))

새 서버가 준비되면 아래 셀을 실행한다. 두 답변의 정답과 종료 이유를 비교한다. `stop`은 모델의 종료 조건, `length`는 출력 상한에 도달했다는 뜻이다. 한 문항에서 답이 맞았다는 관찰로 전체 답변의 품질을 판단할 수는 없다.

In [ ]:
dflash_answer = ask(check_question)
pd.DataFrame([baseline_answer, dflash_answer], index=["Gemma 단독", "DFlash"])[
    ["답변", "종료", "출력 토큰"]
]

## 6. 두 구성의 처리량과 채택률 비교

**채택률**은 제안한 후보 중 Gemma가 받아들인 토큰의 비율이다. 예를 들어 100개를 제안하고 60개를 받아들였으면 60%다. vLLM의 `/metrics`는 서버 시작 후 누적된 수를 돌려준다. 측정 직전과 직후의 차이를 사용한다. 이 비교 동안에는 이 서버를 노트북 하나에서만 사용한다.

워밍업을 측정 구간에서 빼기 위해 아래에서는 초안 카운터도 워밍업 직후에 읽는다. [vLLM 초안 토큰 카운터](https://docs.vllm.ai/en/v0.34.0/features/speculative_decoding/acceptance_metrics/)

In [ ]:
def draft_counts():
    response = requests.get(f"{base_url}/metrics")
    response.raise_for_status()
    names = {
        "vllm:spec_decode_num_draft_tokens_total": "제안",
        "vllm:spec_decode_num_accepted_tokens_total": "채택",
    }
    counts = {"제안": 0, "채택": 0}
    for family in text_string_to_metric_families(response.text):
        for sample in family.samples:
            if sample.name in names:
                counts[names[sample.name]] += int(sample.value)
    return counts

In [ ]:
dflash = pd.DataFrame([measure(1, draft_counts), measure(4, draft_counts)]).set_index("동시 요청")
dflash.round(2)

먼저 **제안 토큰**이 0보다 큰지 확인한다. 0이거나 채택률이 비어 있으면 새 서버가 DFlash를 사용 중인지 시작 로그의 `SpeculativeConfig`와 `/metrics`의 카운터를 확인한다. 그런 경우에는 처리량 차이를 DFlash의 효과로 해석할 수 없다.

다음 표에서 **처리량 배수**는 같은 동시 요청 수에서 DFlash 처리량을 단독 처리량으로 나눈 값이다. 1보다 크면 그 조건에서 서버 전체가 초당 더 많은 토큰을 생성했다. **TPOT 감소율**은 양수일 때 토큰 하나를 더 만드는 평균 시간이 줄었음을 뜻한다. 응답 길이가 달라졌다면 두 표의 출력 토큰 합도 함께 읽는다.

In [ ]:
comparison = pd.DataFrame({
    "단독 토큰/초": baseline["출력 토큰/초"],
    "DFlash 토큰/초": dflash["출력 토큰/초"],
    "처리량 배수": dflash["출력 토큰/초"] / baseline["출력 토큰/초"],
    "TPOT 감소율 %": 100 * (1 - dflash["TPOT p50 ms"] / baseline["TPOT p50 ms"]),
    "단독 응답 p95 ms": baseline["응답 p95 ms"],
    "DFlash 응답 p95 ms": dflash["응답 p95 ms"],
    "채택률 %": dflash["채택률 %"],
})
comparison.round(2)

### 결과 읽기와 종료

1. **혼자 사용할 때** 동시 요청 1의 TPOT 감소율과 응답 p95를 함께 읽는다. 토큰당 시간은 줄었는데 전체 시간 차이가 작다면, 실제 출력 길이와 큐 대기·TTFT도 확인한다.
2. **여러 사람이 사용할 때** 동시 요청 4의 처리량 배수와 응답 p95를 읽는다. 여러 요청을 함께 처리하면 단독 서버도 GPU를 더 많이 사용할 수 있어, 동시 요청 1에서 본 배수가 달라질 수 있다.
3. **초안이 얼마나 쓰였는지** 제안 토큰 수, 채택률, TPOT를 함께 본다. 채택률이 높아도 초안 계산 시간과 검증 비용이 들기 때문에 속도 변화는 실제 측정으로 판단한다.

토론할 때는 “동시 요청 __개에서 단독은 __토큰/초, DFlash는 __토큰/초였다. 같은 조건의 응답 p95는 __ms에서 __ms로 변했다”처럼 조건과 단위를 함께 말한다. 측정을 마치면 **서버 터미널에서 Ctrl+C**를 눌러 GPU를 반환한다.

입력 길이를 따로 바꾸는 실험은 [H100 부하 테스트 노트](h100.ipynb)의 6절, 출력 길이를 바꾸는 실험은 같은 노트의 7절에 있다.